# TabPFN sobre o corpus completo (GPU) — hate-speech EN/PT

Teste justo do TabPFN: **amostra toda** (sem subamostrar no CPU) na **GPU**, sobre as
features que a gente usa. Compara, nas MESMAS features, **TabPFN vs LightGBM vs LogReg**,
e ainda dá ao TabPFN uma segunda base: **TF-IDF reduzido por SVD** (carrega o sinal de
superfície que o SBERT perde).

Referência do leaderboard: `tfidf_logreg` strict **0.709** · `xlmr` strict **0.750** · `sbert_lgbm` strict 0.683.

## Como usar
1. **Runtime → Change runtime type → GPU** (T4 serve; A100 aguenta mais contexto).
2. Rode as células em ordem. Cole sua API key do TabPFN (ux.priorlabs.ai) quando pedir.
3. Suba `corpus_strict.parquet` e `corpus_broad.parquet` (pasta `data/processed/` do seu PC).
4. No fim baixe `tabpfn_results.zip` e me mande. Eu mesclo no leaderboard local (McNemar, tabelas).

## 1 · GPU

In [ ]:
import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("SEM GPU — Runtime > Change runtime type > GPU, e rode de novo")

## 2 · Instalar

In [ ]:
!pip install -q tabpfn "sentence-transformers>=2.7" "scikit-learn>=1.4" lightgbm pyarrow

## 3 · API key do TabPFN (não fica salva no notebook)

In [ ]:
import os, getpass
os.environ["TABPFN_TOKEN"] = getpass.getpass("TabPFN API key (ux.priorlabs.ai): ").strip()
print("token definido:", bool(os.environ.get("TABPFN_TOKEN")))

## 4 · Subir o corpus + trava anti-mojibake\nSuba `corpus_strict.parquet` E `corpus_broad.parquet`.

In [ ]:
import pandas as pd
from google.colab import files
up = files.upload()
CORPUS = {}
for name in up:
    pol = "strict" if "strict" in name else "broad" if "broad" in name else None
    if pol:
        CORPUS[pol] = pd.read_parquet(name)
assert "strict" in CORPUS and "broad" in CORPUS, "suba os DOIS parquets (strict e broad)"
for pol, c in CORPUS.items():
    bad = c["text_clean"].astype(str).str.contains("Ã©|Ã£|Ã§|Ã³").mean()
    print(pol, "| linhas:", len(c), "| suspeita de mojibake:", f"{bad:.4%}")
    assert bad < 0.01, f"{pol}: texto parece latin-1 corrompido (PT tem que ser UTF-8)" 

## 5 · Helpers (métricas + embeddings)

In [ ]:
import numpy as np, time
from sklearn.metrics import f1_score, roc_auc_score, recall_score
from sentence_transformers import SentenceTransformer

def macro_f1(y, p): return f1_score(y, p, average="macro")
def best_thr(y, s):
    g = np.linspace(0.05, 0.95, 181)
    return g[int(np.argmax([macro_f1(y, (s >= t).astype(int)) for t in g]))]
def ev(y, s, thr):
    p = (s >= thr).astype(int)
    return dict(f1=macro_f1(y, p), auc=roc_auc_score(y, s), rec_hate=recall_score(y, p, pos_label=1))

def split_xy(df, sp):
    s = df[df["split"] == sp].dropna(subset=["text_clean"]).copy()
    return s, s["label"].to_numpy()

_ENC = None
def embed(texts):
    global _ENC
    if _ENC is None:
        _ENC = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device="cuda")
    return _ENC.encode(list(texts), batch_size=256, convert_to_numpy=True, show_progress_bar=True)

## 6 · Runner: TabPFN (amostra toda, com fallback de memória) + baselines nas MESMAS features

In [ ]:
from tabpfn import TabPFNClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(42)
ROWS, PREDS = [], {}

def fit_tabpfn(Xtr, ytr, Xv, Xte):
    # usa a amostra toda; se a GPU estourar, reduz o contexto e avisa
    for cap in [len(ytr), 15000, 8000, 4000]:
        try:
            if cap < len(ytr):
                idx = rng.choice(len(ytr), cap, replace=False); xt, yt = Xtr[idx], ytr[idx]
            else:
                xt, yt = Xtr, ytr
            clf = TabPFNClassifier(device="cuda", ignore_pretraining_limits=True, random_state=42)
            clf.fit(xt, yt)
            print(f"    tabpfn: treino={len(yt)}")
            return clf.predict_proba(Xv)[:, 1], clf.predict_proba(Xte)[:, 1]
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                torch.cuda.empty_cache(); print(f"    OOM em {cap}, reduzindo contexto…"); continue
            raise
    raise RuntimeError("tabpfn: OOM mesmo no menor contexto")

def run_features(tag, feats):
    for pol in ["strict", "broad"]:
        (Xtr, ytr, tr), (Xv, yv, _), (Xte, yte, te) = feats[pol]
        print(f"\n[{tag} · {pol}] train={len(ytr)} val={len(yv)} test={len(yte)}")
        t0 = time.time()
        try:
            sv, ste = fit_tabpfn(Xtr, ytr, Xv, Xte)
            thr = best_thr(yv, sv); r = ev(yte, ste, thr)
            ROWS.append(dict(features=tag, policy=pol, model="tabpfn", **r, secs=round(time.time()-t0)))
            print(f"  tabpfn   F1={r['f1']:.4f} AUC={r['auc']:.4f} rec_hate={r['rec_hate']:.4f} ({time.time()-t0:.0f}s)")
            mid = f"tabpfn_{tag}_{pol}_s42"
            PREDS[mid] = te.assign(y_true=yte, y_score=ste, y_pred=(ste >= thr).astype(int))[
                ["id", "language", "source_dataset", "y_true", "y_score", "y_pred"]]
        except Exception as e:
            print("  tabpfn FALHOU:", type(e).__name__, e)
        for mname, clf in [("lgbm", LGBMClassifier(n_estimators=400, random_state=42, class_weight="balanced")),
                           ("logreg", LogisticRegression(max_iter=2000, class_weight="balanced"))]:
            clf.fit(Xtr, ytr); sv = clf.predict_proba(Xv)[:, 1]; ste = clf.predict_proba(Xte)[:, 1]
            thr = best_thr(yv, sv); r = ev(yte, ste, thr)
            ROWS.append(dict(features=tag, policy=pol, model=mname, **r, secs=0))
            print(f"  {mname:8s} F1={r['f1']:.4f} AUC={r['auc']:.4f} rec_hate={r['rec_hate']:.4f}")

## 7 · Base A: embeddings SBERT (mesmas features do sbert_lgbm)

In [ ]:
def build_sbert():
    feats = {}
    for pol in ["strict", "broad"]:
        df = CORPUS[pol]; packs = []
        for sp in ["train", "val", "test"]:
            s, y = split_xy(df, sp)
            packs.append((embed(s["text_clean"]), y, s))
        feats[pol] = tuple(packs)
    return feats

run_features("sbert", build_sbert())

## 8 · Base B: TF-IDF (word+char) reduzido por SVD para 300 dims\nDá ao TabPFN o sinal lexical de superfície, em forma densa e de baixa dimensão.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import hstack

def build_tfidf_svd(dim=300):
    feats = {}
    for pol in ["strict", "broad"]:
        df = CORPUS[pol]
        tr, ytr = split_xy(df, "train"); va, yv = split_xy(df, "val"); te, yte = split_xy(df, "test")
        word = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=3, sublinear_tf=True)
        char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, sublinear_tf=True)
        Xtr = hstack([word.fit_transform(tr["text_clean"]), char.fit_transform(tr["text_clean"])]).tocsr()
        Xv  = hstack([word.transform(va["text_clean"]),     char.transform(va["text_clean"])]).tocsr()
        Xte = hstack([word.transform(te["text_clean"]),     char.transform(te["text_clean"])]).tocsr()
        svd = TruncatedSVD(n_components=dim, random_state=42).fit(Xtr)
        feats[pol] = ((svd.transform(Xtr), ytr, tr), (svd.transform(Xv), yv, va), (svd.transform(Xte), yte, te))
    return feats

run_features("tfidf_svd300", build_tfidf_svd(300))

## 9 · Resultado + zip para me mandar

In [ ]:
import json, zipfile, os
res = pd.DataFrame(ROWS).sort_values(["features", "policy", "f1"], ascending=[True, True, False])
print(res.to_string(index=False))
print("\nReferência: tfidf_logreg strict 0.709 · xlmr strict 0.750 · sbert_lgbm strict 0.683")
os.makedirs("reports/predictions", exist_ok=True)
for mid, df in PREDS.items():
    df.to_parquet(f"reports/predictions/{mid}_test.parquet", index=False)
res.to_json("tabpfn_results.json", orient="records", indent=2)
with zipfile.ZipFile("tabpfn_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("tabpfn_results.json")
    for f in os.listdir("reports/predictions"):
        if f.startswith("tabpfn_"):
            z.write(f"reports/predictions/{f}", f"reports/predictions/{f}")
from google.colab import files
files.download("tabpfn_results.zip")
print("pronto — me manda o tabpfn_results.zip")